In [ ]:
import pandas as pd
# Install spacy
## -q is for "quiet". It removes logs
%pip install spacy -q
# Import spacy, if you don't have it --> !pip install spacy
import spacy
spacy.__version__
# Download model
!python -m spacy download en_core_web_sm -q
# Import english stop words
from spacy.lang.en.stop_words import STOP_WORDS
import re

In [ ]:
data = pd.read_csv(r'C:\Users\LN\Documents\Jedha\Projets\NORTH_FACE\Helene\data\sample_clean.csv')

In [ ]:
data.head()

In [ ]:
type(data)

In [ ]:
data['products'].tolist()

In [ ]:
cols = ["products", 'advantages', 'details', 'fabric', 'weight', 'country', 'description_text']
[data[c].str.lower() for c in cols]

In [ ]:
data['text'] = data['products'] + ' ' + data['advantages']

In [ ]:
data.head(1)

In [ ]:
import en_core_web_sm
nlp = en_core_web_sm.load() 
docs = list(nlp.pipe(data['text'].astype(str)))
for doc in docs:

# Print tokens
    print("Tokens: {}".format([token.text for token in doc]))
    print("------------")

    # Print Lemmatized tokens
    print("Lemma: {}".format([token.lemma_ for token in doc]))
    print("------------")

    # Print Indices of each tokens
    print("Indices: {}".format([token.i for token in doc]))
    print("------------")

    # Print whether tokens are alpha numeric
    print("Alpha Numeric: {}".format([token.is_alpha for token in doc]))
    print("------------")

    # Print whether tokens are numbers
    print("Numbers: {}".format([token.like_num for token in doc]))

# # Ajouter des stop words personnalisés
# nlp.Defaults.stop_words.update({"and", "the", "a", "for", "in", "to", "of", "is", "on", "from"})

# # Appliquer les modifications au vocabulaire
# for word in {"and", "the", "a", "for", "in", "to", "of", "is", "on", "from"}:
#     nlp.vocab[word].is_stop = True

In [ ]:
import re
# Import matplotlib and wordcloud
import matplotlib.pyplot as plt
import wordcloud
from spacy.lang.en.stop_words import STOP_WORDS
from spacy.lang.char_classes import HYPHENS
from spacy.util import compile_infix_regex

infixes = [x for x in nlp.Defaults.infixes if HYPHENS not in x]
nlp.tokenizer.infix_finditer = compile_infix_regex(infixes).finditer

text = " ".join(data['text'].astype(str))
text = re.sub(r"<[^>]+>", " ", text) # replace html
# Garder uniquement les lettres anglaises, espaces et tirets
text = re.sub(r"[^a-z\s-]+", " ", text.lower())

# Garder uniquement les tirets entre deux lettres
text = re.sub(r"(?<![a-z])-|-(?![a-z])", " ", text)

# Normaliser les espaces
text = re.sub(r"\s+", " ", text).strip()

doc = nlp(text)
lemma = [token.lemma_ for token in doc if token.lemma_ not in STOP_WORDS]
lemma

cloud = wordcloud.WordCloud(
    regexp=r"[\w'-]+", # MODIF : garde les mots avec tiret
    stopwords=STOP_WORDS, 
    collocations=True, # remove bigrams
    min_font_size=7,
    background_color="white", 
    colormap="viridis"
).generate(" ".join(lemma)) # remove stopwords

# Show word cloud with matplotlib
plt.imshow(cloud)
plt.axis("off")

# Remove trailing logs
plt.show()

In [ ]:
data.head()

In [ ]:
from collections import Counter; Counter(lemma).most_common()

In [ ]:
pd.set_option("display.max_rows", None)      # show all rows
pd.set_option("display.max_colwidth", None)  # no column truncation

In [ ]:
data[["products", "advantages"]].drop_duplicates(subset="products")
data.head(1)

In [ ]:
def lemma(doc):
    return "".join(token.lemma_ + token.whitespace_ for token in doc).strip()

keywords = [
    'bag',
    'baggage',
    # 'belt',
    'boot',
    'bottom | btm',
    'boxer',
    'bra',
    'brief',
    # 'cap',
    'capri',
    'cardigan',
    'daypack',
    'divider',
    'dress',
    'duffle',
    'glove',
    'hat',
    'hoody',
    'jeans-',
    'jacket | jkt', 
    'pant',
    'polo',
    'poster',
    'pullover | p/o',
    'rashguard',
    'short',
    'skirt',
    'sneaker',
    'sock',
    'sun mask' # anti bronzage sur le visage
    'storage locker',
    'sweater',
    'sweatshirt',
    'tank',
    'top',
    'tote',
    't-shirt | t-shir',
    'shirt',
    'tight',
    'trench coat',
    'surf trunk', # short de bain
    'underwear',
    'vest',
    'visor',
    'wader',
    'zip neck'
    ]
data["lemma"] = [lemma(doc) for doc in docs]
keywords_lemma = [lemma(doc) for doc in nlp.pipe(keywords)]
data["product_keyword"] = ""

for groupe in keywords:
    treatment = [
        re.escape(word) + (r"s?" if word[-1].isalpha() and not word.endswith("s") else "")
        for word in map(str.strip, groupe.split("|")) if word
    ]
    pattern = r"(?<!\w)(?:" + "|".join(treatment) + r")(?!\w)" if treatment else r"(?!)"
    search = data["lemma"].str.contains(pattern, case=False, na=False)
    data.loc[search, "product_keyword"] += groupe.split("|")[0].strip() + ", "

data["product_keyword"] = data["product_keyword"].str.removesuffix(", ")
search = data["product_keyword"] != ""
not_found = data.loc[~search, ["lemma", "products", "advantages"]]
print(f"{len(not_found)} documents sans mot-clé sur {len(data)}")
not_found.head(1)

In [ ]:
data[['lemma', 'products', 'product_keyword']]

La liste est très bruitée. On va se recentrer sur les produits maintenant que nous les connaissons.

In [ ]:
data['text'] = data['products']


In [ ]:
import en_core_web_sm
nlp = en_core_web_sm.load() 
docs = list(nlp.pipe(data['text'].astype(str)))
for doc in docs:

# Print tokens
    print("Tokens: {}".format([token.text for token in doc]))
    print("------------")

    # Print Lemmatized tokens
    print("Lemma: {}".format([token.lemma_ for token in doc]))
    print("------------")

    # Print Indices of each tokens
    print("Indices: {}".format([token.i for token in doc]))
    print("------------")

    # Print whether tokens are alpha numeric
    print("Alpha Numeric: {}".format([token.is_alpha for token in doc]))
    print("------------")

    # Print whether tokens are numbers
    print("Numbers: {}".format([token.like_num for token in doc]))

# # Ajouter des stop words personnalisés
# nlp.Defaults.stop_words.update({"and", "the", "a", "for", "in", "to", "of", "is", "on", "from"})

# # Appliquer les modifications au vocabulaire
# for word in {"and", "the", "a", "for", "in", "to", "of", "is", "on", "from"}:
#     nlp.vocab[word].is_stop = True

In [ ]:
import re
# Import matplotlib and wordcloud
import matplotlib.pyplot as plt
import wordcloud
from spacy.lang.en.stop_words import STOP_WORDS
from spacy.lang.char_classes import HYPHENS
from spacy.util import compile_infix_regex

infixes = [x for x in nlp.Defaults.infixes if HYPHENS not in x]
nlp.tokenizer.infix_finditer = compile_infix_regex(infixes).finditer

text = " ".join(data['text'].astype(str))
text = re.sub(r"<[^>]+>", " ", text) # replace html
# Garder uniquement les lettres anglaises, espaces et tirets
text = re.sub(r"[^a-z\s-]+", " ", text.lower())

# Garder uniquement les tirets entre deux lettres
text = re.sub(r"(?<![a-z])-|-(?![a-z])", " ", text)

# Normaliser les espaces
text = re.sub(r"\s+", " ", text).strip()

doc = nlp(text)
lemma = [token.lemma_ for token in doc if token.lemma_ not in STOP_WORDS]
lemma

cloud = wordcloud.WordCloud(
    regexp=r"[\w'-]+", # MODIF : garde les mots avec tiret
    stopwords=STOP_WORDS, 
    collocations=True, # remove bigrams
    min_font_size=7,
    background_color="white", 
    colormap="viridis"
).generate(" ".join(lemma)) # remove stopwords

# Show word cloud with matplotlib
plt.imshow(cloud)
plt.axis("off")

# Remove trailing logs
plt.show()

In [ ]:
data["products"].drop_duplicates()
data.head(1)

In [ ]:
def lemma(doc):
    return "".join(token.lemma_ + token.whitespace_ for token in doc).strip()

keywords = [
    'all-time shell', # imperméable	
    'backpack | crosstown | atom',
    'bag | minimass | half mass | critical mass | single shot  | lightwire | double haul',
    'baggage',
    'belt',
    'boot | marlwalker',
    'bottom | btm',
    'boxer',
    'bra',
    r"brief | re:(?<!lightwire )brief?",
    r"vest | re:(?<!riverwalker )vest?",
    r"boot | re:boots?|marlwalker?(?! pant?\b)",
    # 'brief',
    'beach bucket',
    'cami', # caraco
    # 'cap',
    'cardigan | versatiliti cardi',
    'divider',
    'dress',
    'duffle | duffel',
    'fleece | synch marsupial', # polaire
    'glove',
    'hat',
    'bimini cap', # casquette 
    'velocity cap', # casquette
    'hip chest pack',
    'hip pack',
    'hipster',
    'hoody',
    r"jeans | re:jeans?-\w+", # rechercher un mot en début de chaîne de caractères
    'jacket | jkt | reversible phone home', 
    'briefcase|lightwire brief',
    'carryon| mlc',
    'mat | remat',
    'pant | capri', # capri = pantacourt
    'pocket pack',
    'polo',
    "poster | traverse auguille d'entreves",
    'pullover | p/o | cap 2 v-neck | synch snap-t | merino 1 crew | merino 2 crew | cap 1 crew | cap 2 crew | cap 3 crew',
    'rashguard',
    'skirt',
    'sneaker',
    'sock',
    'storage locker',
    'sweater',
    'sweatshirt',
    'sweater'
    'shell pant', # pantalon isolant
    'shirt',  
    "bathing suit | girl's two-piece set",
    'bikini',
    'tankini',
    'sun mask',
    r"surf trunk | re:surf trunks?",
    r"short | re:shorts?(?:-\d+)?|shorties?|long haul runner", # espaces recherchés comme tels
    'shoulder bag | daily grind',
    't-back',
    't-shirt | t-shir | cap 1 graphic tee | cap 1 graphic crew | s/s stretch t | versatili-tee | versatiliti tee | simply organic tee | organic cotton under tee | l/s gravi-tee | vitaliti vee',
    'tank', # débardeur
    'tight',
    'top | solid bibiana 1 piece | cap 2 cap sleeve | cap 1 scoop',
    'tote',
    'travel courier',
    'travel pack',
    'travel sling',
    'trench coat',
    'tunic',
    'vest | aravis 1/4 zip',
    'visor | brim',
    'wader | riverwalker | watermaster',
    'waterproof pack | stormfront pack',
    'wheeled bag | freewheel',
    'wrap', # écharpe
    'zip neck'
    ]
data["lemma"] = [lemma(doc) for doc in docs]
keywords_lemma = [lemma(doc) for doc in nlp.pipe(keywords)]
data["product_keyword"] = ""

import re
positions = pd.Series(-1, index=data.index)

# pour utiliser des regex
for groupe in keywords:
    if " | re:" in groupe:
        treatment = [groupe.split(" | re:", 1)[1]]
    else:
        treatment = [
            word + (r"s?" if word[-1].isalpha() and not word.endswith("s") else "")
            for word in map(str.strip, groupe.split("|")) if word
        ]
    # pattern = r"(?<!\w)(?:" + "|".join(treatment) + r")(?!\w)" if treatment else r"(?!)"
    pattern = r"(?<![\w-])(?:" + "|".join(treatment) + r")(?![\w-])" if treatment else r"(?!)"
    search = data["lemma"].str.contains(pattern, na=False)
    for i, texte in data.loc[search, "lemma"].items():
            for trouve in re.finditer(pattern, texte):
                debut = trouve.start()
                if debut > positions.loc[i]:
                    positions.loc[i] = debut
                    data.loc[i, "product_type"] = groupe.split("|")[0].strip()

data["product_keyword"] = data["product_keyword"].str.removesuffix(", ")
search = data["product_keyword"] != ""
not_found = data.loc[~search, ["lemma", "products"]]
print(f"{len(not_found)} documents sans mot-clé sur {len(data)}")
not_found

In [ ]:
data

In [ ]:
data['product_keyword'].isnull().sum()